# Library Versions

In [2]:
# Install the compatible pair for your new Colab runtime
!pip install tensorflow==2.16.1
!pip install tensorflow-io==0.37.1
!pip install ml_dtypes==0.5.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 589.9/589.9 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 111.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 142.0 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.19.0
    Uninstalling tensorboard-2.19.0:
      Successfully uninstalled tensorboard-2.19.0
  Attempting uninstall: ml-dtypes
    

# Access GCS Bucket

In [ ]:
# Run this cell first

import os
from google.colab import userdata
import json

# --- 1. Securely get the key from Colab Secrets ---
key_string = userdata.get('GCP_KEY')
key_data = json.loads(key_string)

# --- 2. Authenticate gsutil ---
# Write the key to a temporary file that gsutil can read
with open('temp_key.json', 'w') as f:
    json.dump(key_data, f)

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = 'temp_key.json'
!gcloud auth activate-service-account --key-file=temp_key.json

# --- 3. Access bucket directories ---
BUCKET_NAME = "asvspoof-la-data-bucket"
!gsutil ls gs://{BUCKET_NAME}

Activated service account credentials for: [colab-storage-reader@voiceauth-479800.iam.gserviceaccount.com]
gs://asvspoof-la-data-bucket/ASVspoof2019_LA_cm_protocols/
gs://asvspoof-la-data-bucket/ASVspoof2019_LA_dev/
gs://asvspoof-la-data-bucket/ASVspoof2019_LA_eval/
gs://asvspoof-la-data-bucket/ASVspoof2019_LA_train/
gs://asvspoof-la-data-bucket/tfrecords/


# Datasets

In [ ]:
import pandas as pd
import tensorflow as tf
import tensorflow_io as tfio
import numpy as np
import json
import os
from google.colab import userdata
from tqdm.auto import tqdm # For a progress bar!

# --- 1. Authentication & Setup ---
# This cell is self-contained. It re-creates the key file
# for the GCS write operations.
KEY_FILE_NAME = 'temp_key.json'
try:
    key_string = userdata.get('GCP_KEY')
    key_data = json.loads(key_string)
    with open(KEY_FILE_NAME, 'w') as f:
        json.dump(key_data, f)

    # Set the environment variable for TF
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = KEY_FILE_NAME
    # Activate gcloud for shell commands (just in case)
    !gcloud auth activate-service-account --key-file={KEY_FILE_NAME}
    print("✅ Auth activated.")
except Exception as e:
    print(f"⚠️ Auth failed, but may already be active: {e}")

storage_options = {'token': KEY_FILE_NAME}

# --- 2. Define All Constants ---
SAMPLE_RATE = 16000
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
BUCKET_NAME = "asvspoof-la-data-bucket"
PROTOCOL_DIR = "ASVspoof2019_LA_cm_protocols"
AUDIO_DIR = "ASVspoof2019_LA_train"
DEV_AUDIO_DIR = "ASVspoof2019_LA_dev"
EVAL_AUDIO_DIR = "ASVspoof2019_LA_eval" # <-- ADDED FOR EVAL

# --- 3. Load Label Files ---
print("Loading label files from GCS...")
column_names = ['SPEAKER_ID', 'AUDIO_FILE_NAME', 'BLANK', 'SYSTEM_ID', 'KEY']

# Load TRAINING Labels
train_protocol_path = f"gs://{BUCKET_NAME}/{PROTOCOL_DIR}/ASVspoof2019.LA.cm.train.trn.txt"
train_df = pd.read_csv(
    train_protocol_path, sep=' ', header=None,
    names=column_names, storage_options=storage_options
)
train_df = train_df[['AUDIO_FILE_NAME', 'KEY']]
train_df['LABEL'] = train_df['KEY'].apply(lambda x: 0 if x == 'bonafide' else 1)
train_df = train_df.drop(columns=['KEY'])
print(f"✅ Loaded {len(train_df)} training labels.")

# Load VALIDATION Labels
dev_protocol_path = f"gs://{BUCKET_NAME}/{PROTOCOL_DIR}/ASVspoof2019.LA.cm.dev.trl.txt"
val_df = pd.read_csv(
    dev_protocol_path, sep=' ', header=None,
    names=column_names, storage_options=storage_options
)
val_df = val_df[['AUDIO_FILE_NAME', 'KEY']]
val_df['LABEL'] = val_df['KEY'].apply(lambda x: 0 if x == 'bonafide' else 1)
val_df = val_df.drop(columns=['KEY'])
print(f"✅ Loaded {len(val_df)} validation labels.")

# Load EVALUATION Labels <-- ADDED
eval_protocol_path = f"gs://{BUCKET_NAME}/{PROTOCOL_DIR}/ASVspoof2019.LA.cm.eval.trl.txt"
eval_df = pd.read_csv(
    eval_protocol_path, sep=' ', header=None,
    names=column_names, storage_options=storage_options
)
eval_df = eval_df[['AUDIO_FILE_NAME', 'KEY']]
eval_df['LABEL'] = eval_df['KEY'].apply(lambda x: 0 if x == 'bonafide' else 1)
eval_df = eval_df.drop(columns=['KEY'])
print(f"✅ Loaded {len(eval_df)} evaluation labels.")


# --- 4. Pre-compute Mel Filterbank ---
num_spectrogram_bins = N_FFT // 2 + 1
mel_filterbank = tf.signal.linear_to_mel_weight_matrix(
    num_mel_bins=N_MELS,
    num_spectrogram_bins=num_spectrogram_bins,
    sample_rate=SAMPLE_RATE,
    lower_edge_hertz=0.0,
    upper_edge_hertz=(SAMPLE_RATE / 2.0)
)
print("✅ Mel filterbank pre-computed.")

Activated service account credentials for: [colab-storage-reader@voiceauth-479800.iam.gserviceaccount.com]
✅ Auth activated.
Loading label files from GCS...
✅ Loaded 25380 training labels.
✅ Loaded 24844 validation labels.
✅ Loaded 71237 evaluation labels.
✅ Mel filterbank pre-computed.


In [ ]:
# --- 5. Define Processing Function (Python-native version) ---
def load_and_preprocess_eager(audio_file_name, gcs_audio_dir):
    # Use f-string, as this is eager Python
    gcs_path = f"gs://{BUCKET_NAME}/{gcs_audio_dir}/flac/{audio_file_name}.flac"

    audio_binary = tf.io.read_file(gcs_path)
    waveform = tfio.audio.decode_flac(audio_binary, dtype=tf.int16)
    waveform = tf.cast(waveform, tf.float32) / 32768.0
    waveform = tf.squeeze(waveform, axis=-1)

    stft = tf.signal.stft(waveform, frame_length=N_FFT, frame_step=HOP_LENGTH, fft_length=N_FFT)
    spectrogram = tf.abs(stft)

    mel_spectrogram = tf.tensordot(spectrogram, mel_filterbank, 1)
    log_mel_spectrogram = tf.math.log(mel_spectrogram + 1e-6)

    # Set dynamic shape. This is important for variable-length audio.
    log_mel_spectrogram.set_shape([None, N_MELS])

    return log_mel_spectrogram

# --- 6. Define TFRecord Helper Functions ---
def _bytes_feature(value):
    """Returns a bytes_list from a string / byte."""
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

def _int64_feature(value):
    """Returns an int64_list from a bool / enum / int / uint."""
    return tf.train.Feature(int64_list=tf.train.Int64List(value=[value]))

def serialize_example(spectrogram, label):
    """
    Creates a tf.train.Example message ready to be written to a file.
    """
    spec_string = tf.io.serialize_tensor(spectrogram)

    feature = {
        'spectrogram': _bytes_feature(spec_string.numpy()),
        'label': _int64_feature(label)
    }

    example_proto = tf.train.Example(features=tf.train.Features(feature=feature))
    return example_proto.SerializeToString()


# --- 7. Define Main Writer Function ---
def process_and_write_tfrecords(df, gcs_audio_dir, gcs_output_path):
    """
    Loops through a DataFrame, processes audio, and writes to a TFRecord file on GCS.
    """
    print(f"\nStarting to process {len(df)} files...")
    print(f"Writing to: {gcs_output_path}")

    with tf.io.TFRecordWriter(gcs_output_path) as writer:
        for _, row in tqdm(df.iterrows(), total=len(df)):
            file_name = row['AUDIO_FILE_NAME']
            label = row['LABEL']

            try:
                spectrogram = load_and_preprocess_eager(file_name, gcs_audio_dir)
                example = serialize_example(spectrogram, label)
                writer.write(example)
            except Exception as e:
                print(f"\nWARNING: Skipping file {file_name}. Error: {e}")

    print(f"\n✅ Successfully created {gcs_output_path}")

# --- 8. RUN THE CONVERSION ---

# Define output paths
TFRECORD_DIR = "tfrecords"
GCS_TRAIN_OUTPUT_PATH = f"gs://{BUCKET_NAME}/{TFRECORD_DIR}/train.tfrecord"
GCS_VAL_OUTPUT_PATH = f"gs://{BUCKET_NAME}/{TFRECORD_DIR}/val.tfrecord"
GCS_EVAL_OUTPUT_PATH = f"gs://{BUCKET_NAME}/{TFRECORD_DIR}/eval.tfrecord" # <-- ADDED

### Train

In [ ]:
# Process the Training Set
process_and_write_tfrecords(
    df=train_df,
    gcs_audio_dir=AUDIO_DIR,
    gcs_output_path=GCS_TRAIN_OUTPUT_PATH
)


Starting to process 25380 files...
Writing to: gs://asvspoof-la-data-bucket/tfrecords/train.tfrecord


  0%|          | 0/25380 [00:00<?, ?it/s]

### Validation

In [ ]:
# Process the Validation Set
process_and_write_tfrecords(
    df=val_df,
    gcs_audio_dir=DEV_AUDIO_DIR,
    gcs_output_path=GCS_VAL_OUTPUT_PATH
)

### Test

In [ ]:
# Process the Evaluation Set <-- ADDED
process_and_write_tfrecords(
    df=eval_df,
    gcs_audio_dir=EVAL_AUDIO_DIR,
    gcs_output_path=GCS_EVAL_OUTPUT_PATH
)

### Cleanup

In [ ]:
# --- 9. Clean up ---
!rm {KEY_FILE_NAME}
print("\n🎉 All processing complete. Key file removed.")

# CNN

# RNN

In [1]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Bidirectional, Masking, Dropout

# --- 1. Define Model Parameters ---
# Input shape: (Timesteps, Mel_bins)
# We use 'None' for timesteps because of padding
# We use 'N_MELS' (128) for the features
INPUT_SHAPE = (None, N_MELS)

# --- 2. Build the Model Architecture ---
model = Sequential([
    # This is the crucial layer we discussed.
    # It tells the model to ignore any timesteps that are all zeros.
    Masking(mask_value=0.0, input_shape=INPUT_SHAPE),

    # A Bidirectional GRU layer. 'return_sequences=True' means it outputs
    # the full sequence for the next layer to read.
    Bidirectional(GRU(64, return_sequences=True)),

    # A second Bidirectional GRU layer. This one (default) only
    # returns the *final* output after processing the whole sequence.
    Bidirectional(GRU(32)),

    # A standard "classifier" head
    Dense(16, activation='relu'),
    Dropout(0.3), # Dropout helps prevent overfitting

    # The final output layer.
    # '1' unit (for bonafide vs. spoof)
    # 'sigmoid' activation (squeezes the output between 0 and 1)
    Dense(1, activation='sigmoid')
])

# --- 3. Compile the Model ---
# This prepares the model for training
model.compile(
    # Adam is a great, all-purpose optimizer
    optimizer='adam',

    # This is the standard loss function for 0/1 classification
    loss='binary_crossentropy',

    # We want to track accuracy and AUC (Area Under the Curve),
    # which is a great metric for this type of problem.
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# --- 4. Print the Model Summary ---
# This shows you the layers, their shapes, and the number of parameters.
model.summary()

NameError: name 'N_MELS' is not defined

In [ ]:
import tensorflow as tf

# --- 1. Define the GCS Checkpoint Path ---
BUCKET_NAME = "asvspoof-la-data-bucket"
# We'll create a new 'models' folder in your bucket
CHECKPOINT_PATH = f"gs://{BUCKET_NAME}/models/best_model.keras"

print(f"Checkpoints will be saved to: {CHECKPOINT_PATH}")

# --- 2. Define Callbacks ---

# Save the best version of the model to a file
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath=CHECKPOINT_PATH,  # File path to save the model
    save_weights_only=False,    # Save the entire model
    monitor="val_auc",          # The metric to monitor
    mode="max",                 # We want to MAXIMIZE AUC
    save_best_only=True         # Only save when it's the "best"
)

# Stop training early if the model stops improving
early_stopping_cb = tf.keras.callbacks.EarlyStopping(
    monitor="val_auc",          # The metric to monitor
    mode="max",                 # We want to MAXIMIZE AUC
    patience=5,                 # Stop after 5 epochs of no improvement
    restore_best_weights=True   # Restore the weights from the best epoch
)

# --- 3. Define Training Parameters ---
EPOCHS = 20  # We'll set a high number, EarlyStopping will find the best
             # This will take a long time for the first epoch (caching)

# --- 4. Start Training ---
print("Starting model training...")
history = model.fit(
    train_dataset,                    # Your training data pipeline
    epochs=EPOCHS,
    validation_data=val_dataset, # Your validation data pipeline
    callbacks=[checkpoint_cb, early_stopping_cb] # Our two callbacks
)

print("✅ Training complete.")

gs://asvspoof-la-data-bucket/ASVspoof2019_LA_cm_protocols/
gs://asvspoof-la-data-bucket/ASVspoof2019_LA_dev/
gs://asvspoof-la-data-bucket/ASVspoof2019_LA_eval/
gs://asvspoof-la-data-bucket/ASVspoof2019_LA_train/
Checkpoints will be saved to: gs://asvspoof-la-data-bucket/models/best_model.keras
Starting model training...
Epoch 1/20
